# Pretraining FL FedProx mu=0.01 — smoke + pilot (v0.2)

Notebook de ejecucion del **smoke + pilot FedProx** sobre el corpus FL
(36 PRETRAIN_SOURCE, 10 clientes audit v2.3, caps cerrados 0.10/0.25/0.005).
El smoke pre-valida pipeline; el pilot produce el ckpt federado FedProx
que evaluaremos despues en downstream (HSG18/CWRU).

## Que hace

- Smoke FedProx mu=0.01 (2 rondas x 5 steps x 10 clientes = 100 local steps).
- Pilot FedProx mu=0.01 (10 rondas x 50 steps x 10 clientes = 5 000 local steps).
- Mismo presupuesto y misma topologia que el pilot FedAvg cerrado (commit `9b6c9fb`).
- Unica diferencia: termino proximal `0.5 * mu * sum_l ||theta - theta_global||^2`
  por cliente, anclando los updates locales al state_dict global de inicio de ronda.

## Que NO hace

- NO lanza full FL (eso requiere autorizacion explicita y stage=full).
- NO evalua downstream todavia. La evaluacion FedProx downstream
  (HSG18 linear/full + CWRU linear/full con el ckpt_final del FedProx
  pilot) sera un bloque posterior, equivalente al bloque
  `fl_pilot_vs_central` que ya cerramos para FedAvg.
- NO usa el ckpt FedAvg pilot. Usa el ckpt nuevo que produzca esta corrida
  FedProx en `checkpoints/ssl_federated_pilot_fedprox_mu0_01/.../ckpt_final.pt`.
- NO modifica `min_client_presence` ni el sampling policy.

## Restricciones operativas

- Smoke puede ejecutarse en A100 para homogeneidad. Pilot **requiere A100**;
  si la VM no es A100, parar antes de la celda del pilot.
- El pilot tarda ~20 min en A100 (mismo orden que el pilot FedAvg).
- Si el smoke FedProx NO pasa todos los checks, NO lanzar el pilot.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. colab_init

In [ ]:
!bash /content/drive/MyDrive/fm_fl_phmd/colab_init.sh

## 3. Pull al HEAD

Debe incluir los configs FedProx (`ssl_federated_*_fedprox_mu0_01.yaml`),
los tests nuevos y este notebook.

In [ ]:
%cd /content/fm_fl_phmd
!git pull --ff-only
!git log -3 --oneline
!git status --porcelain

## 4. Verificar GPU

Smoke puede correr en A100, T4 o L4. **Pilot requiere A100**. Si no
es A100 cuando llegues a la celda del pilot, **para aqui** y vuelve
con una sesion A100.

In [ ]:
!nvidia-smi | head -15

## 5. pytest preflight

Cubre los tests FL existentes + los nuevos FedProx + configs FedProx.
Si esto falla, **no continuar**.

In [ ]:
!python -m pytest \
  tests/test_fl_aggregation.py \
  tests/test_fl_client_filtering.py \
  tests/test_fl_server_dryrun.py \
  tests/test_adaptive_batch_size.py \
  tests/test_fl_pilot_config.py \
  tests/test_fl_fedprox.py \
  tests/test_fl_fedprox_configs.py -q

## 6. Preparar `_stdout/`

In [ ]:
!mkdir -p /content/drive/MyDrive/fm_fl_phmd/logs/pretraining_federated/_stdout

## 7. Dry-run FedProx smoke

Validacion sin entrenamiento: 10 clientes, 36 PS, 0 TT, forward sintetico OK,
config federated valido (algorithm=fedprox, fedprox_mu=0.01).

In [ ]:
!python -m training.train_ssl_federated \
  --mode dry-run \
  --config training/configs/ssl_federated_smoke_fedprox_mu0_01.yaml \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/pretraining_federated/_stdout/ssl_federated_smoke_fedprox_mu0_01_dryrun.stdout.log

## 8. Smoke FedProx real

100 local steps totales (2 rondas x 5 steps x 10 clientes). Debe
terminar con `smoke_pass=true` y los 6 checks OK. Si algo falla,
**parar aqui** y diagnosticar antes del pilot.

In [ ]:
import time
t0 = time.time()
!python -m training.train_ssl_federated \
  --mode train \
  --config training/configs/ssl_federated_smoke_fedprox_mu0_01.yaml \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/pretraining_federated/_stdout/ssl_federated_smoke_fedprox_mu0_01.stdout.log
print(f"\n[total] smoke FedProx mu=0.01 elapsed = {time.time() - t0:.1f} s")

## 9. Summary smoke

Lectura del `run_info.json` recien generado. Confirma `smoke_pass=true`,
algoritmo=fedprox, mu=0.01, los 6 smoke_checks OK, y los agregados
ponderados separados (reconstruction_loss_mean_weighted vs
fedprox_loss_mean_weighted) que solo aparecen en v0.2.

In [ ]:
import json
from pathlib import Path

run_dir = Path(
    "/content/drive/MyDrive/fm_fl_phmd/logs/pretraining_federated/"
    "ssl_federated_smoke_fedprox_mu0_01_patchtst_phm"
)
ri_path = run_dir / "run_info.json"
assert ri_path.is_file(), f"no encontrado {ri_path}"
ri = json.loads(ri_path.read_text(encoding='utf-8'))

def _fmt(x, fmt='.4f'):
    return ('null' if x is None else format(x, fmt))

print('=== SMOKE FedProx mu=0.01 ===')
print(f"smoke_pass                            : {ri.get('smoke_pass')}")
print(f"algorithm                             : {ri.get('algorithm')}")
print(f"fedprox_mu                            : {ri.get('fedprox_mu')}")
print(f"fedprox_enabled                       : {ri.get('fedprox_enabled')}")
print(f"config_hash                           : {ri.get('config_hash')}")
print(f"git_hash                              : {ri.get('git_hash')}")
print(f"total_local_optimizer_steps           : {ri.get('total_local_optimizer_steps')}")
print(f"max_effective_bc_global               : {ri.get('max_effective_bc_global')}")
print(f"aggregation_weights_policy_effective  : {ri.get('aggregation_weights_policy_effective')}")
print(f"plan_policy_unique                    : {ri.get('plan_policy_unique')}")
print(f"elapsed_seconds                       : {ri.get('elapsed_seconds')}")
print(f"checkpoint_final                      : {ri.get('checkpoint_final')}")

print('\n--- agregados last-round ---')
print(f"final_loss_mean_weighted              : {_fmt(ri.get('final_loss_mean_weighted'))}")
print(f"final_reconstruction_loss_mean_weighted: {_fmt(ri.get('final_reconstruction_loss_mean_weighted'))}")
print(f"final_fedprox_loss_mean_weighted      : {_fmt(ri.get('final_fedprox_loss_mean_weighted'))}")
print(f"final_fedprox_penalty_mean_weighted   : {_fmt(ri.get('final_fedprox_penalty_mean_weighted'))}")

print('\n--- smoke_checks ---')
for k, v in (ri.get('smoke_checks') or {}).items():
    ok = v.get('ok') if isinstance(v, dict) else None
    print(f"  [{ 'OK' if ok else 'FAIL'}] {k}")

# Aborto si smoke_pass != true: NO lanzar pilot.
if not ri.get('smoke_pass'):
    print('\n!!! smoke_pass != true. NO continuar al pilot. Diagnosticar primero.')
else:
    print('\nsmoke OK. Continuar con el dry-run + pilot.')

## 10. Dry-run FedProx pilot

Validacion sin entrenamiento del config pilot. Mismo dry-run que el
smoke, con n_rounds=10 y local_steps=50.

In [ ]:
!python -m training.train_ssl_federated \
  --mode dry-run \
  --config training/configs/ssl_federated_pilot_fedprox_mu0_01.yaml \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/pretraining_federated/_stdout/ssl_federated_pilot_fedprox_mu0_01_dryrun.stdout.log

## 11. Pilot FedProx real (solo si smoke PASS y GPU = A100)

5 000 optimizer steps totales (10 rondas x 50 steps x 10 clientes).
~20 min en A100. Si la GPU no es A100, **NO ejecutar esta celda**.
Si el smoke v0.2 fallo algun check, **NO ejecutar esta celda**.

Esta celda **NO lanza full FL**. El stage=pilot esta gateado por
`STAGE_MAX_LOCAL_STEPS["pilot"] = 20 000`; para full hace falta
`stage=full` en un YAML aparte (no incluido en esta tarea).

In [ ]:
import time
t0 = time.time()
!python -m training.train_ssl_federated \
  --mode train \
  --config training/configs/ssl_federated_pilot_fedprox_mu0_01.yaml \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/pretraining_federated/_stdout/ssl_federated_pilot_fedprox_mu0_01_patchtst_phm.stdout.log
print(f"\n[total] pilot FedProx mu=0.01 elapsed = {time.time() - t0:.1f} s")

## 12. Summary pilot

Lectura del `run_info.json` + `metrics.jsonl` del pilot. Confirma
`pilot_pass=true`, los 6 pilot_checks, y compara la trayectoria de
loss agregada y de reconstruction_loss entre ronda 1 y ronda 10.

In [ ]:
import json
from pathlib import Path

run_dir = Path(
    "/content/drive/MyDrive/fm_fl_phmd/logs/pretraining_federated/"
    "ssl_federated_pilot_fedprox_mu0_01_patchtst_phm"
)
ri_path = run_dir / "run_info.json"
ml_path = run_dir / "metrics.jsonl"
assert ri_path.is_file(), f"no encontrado {ri_path}"
ri = json.loads(ri_path.read_text(encoding='utf-8'))

def _fmt(x, fmt='.4f'):
    return ('null' if x is None else format(x, fmt))

print('=== PILOT FedProx mu=0.01 ===')
print(f"pilot_pass                            : {ri.get('pilot_pass')}")
print(f"algorithm                             : {ri.get('algorithm')}")
print(f"fedprox_mu                            : {ri.get('fedprox_mu')}")
print(f"fedprox_enabled                       : {ri.get('fedprox_enabled')}")
print(f"config_hash                           : {ri.get('config_hash')}")
print(f"git_hash                              : {ri.get('git_hash')}")
print(f"git_dirty                             : {ri.get('git_dirty')}")
print(f"total_local_optimizer_steps           : {ri.get('total_local_optimizer_steps')}")
print(f"max_effective_bc_global               : {ri.get('max_effective_bc_global')}")
print(f"aggregation_weights_policy_effective  : {ri.get('aggregation_weights_policy_effective')}")
print(f"plan_policy_unique                    : {ri.get('plan_policy_unique')}")
print(f"elapsed_seconds                       : {ri.get('elapsed_seconds')}")
print(f"cumulative_communication_mb           : {ri.get('cumulative_communication_mb')}")
print(f"checkpoint_final                      : {ri.get('checkpoint_final')}")

print('\n--- agregados last-round (ronda 10) ---')
print(f"final_loss_mean_weighted              : {_fmt(ri.get('final_loss_mean_weighted'))}")
print(f"final_reconstruction_loss_mean_weighted: {_fmt(ri.get('final_reconstruction_loss_mean_weighted'))}")
print(f"final_fedprox_loss_mean_weighted      : {_fmt(ri.get('final_fedprox_loss_mean_weighted'))}")
print(f"final_fedprox_penalty_mean_weighted   : {_fmt(ri.get('final_fedprox_penalty_mean_weighted'))}")

print('\n--- pilot_checks ---')
for k, v in (ri.get('pilot_checks') or {}).items():
    ok = v.get('ok') if isinstance(v, dict) else None
    print(f"  [{ 'OK' if ok else 'FAIL'}] {k}")

print('\n--- opt_steps_per_client_total ---')
for c, s in sorted((ri.get('opt_steps_per_client_total') or {}).items()):
    print(f"  {c:<20s} {s}")

print('\n--- datasets_seen_by_client ---')
for c, ds in sorted((ri.get('datasets_seen_by_client') or {}).items()):
    print(f"  {c:<20s} {len(ds):>2d}  {ds}")

print('\n--- aggregation_weights_by_client_last_round ---')
for c, w in sorted(
    (ri.get('aggregation_weights_by_client_last_round') or {}).items(),
    key=lambda kv: -kv[1],
):
    print(f"  {c:<20s} {w:.4f}")

# Trayectoria por ronda desde metrics.jsonl
if ml_path.is_file():
    rounds = []
    for line in ml_path.read_text(encoding='utf-8').splitlines():
        try:
            ev = json.loads(line)
        except Exception:
            continue
        if ev.get('kind') == 'round':
            rounds.append(ev)
    if rounds:
        first = rounds[0]; last = rounds[-1]
        print('\n--- trayectoria loss/reconstruction/prox ---')
        print(f"round 1  : loss={_fmt(first.get('loss_mean_weighted'))}  recon={_fmt(first.get('reconstruction_loss_mean_weighted'))}  prox={_fmt(first.get('fedprox_loss_mean_weighted'))}")
        print(f"round {len(rounds)} : loss={_fmt(last.get('loss_mean_weighted'))}  recon={_fmt(last.get('reconstruction_loss_mean_weighted'))}  prox={_fmt(last.get('fedprox_loss_mean_weighted'))}")
        l0 = first.get('loss_mean_weighted'); l1 = last.get('loss_mean_weighted')
        if l0 and l1:
            print(f"loss_delta_pct        : {(l1 - l0)/l0 * 100:+.2f} %")
        r0 = first.get('reconstruction_loss_mean_weighted'); r1 = last.get('reconstruction_loss_mean_weighted')
        if r0 and r1:
            print(f"reconstruction_delta_pct: {(r1 - r0)/r0 * 100:+.2f} %")

# Aborto suave si pilot_pass != true
if not ri.get('pilot_pass'):
    print('\n!!! pilot_pass != true. NO usar este ckpt para downstream sin diagnostico.')
else:
    print('\npilot OK. Siguiente paso (separado): eval downstream HSG18+CWRU con este ckpt FedProx.')

print('\nPega este output completo en el chat para cerrar el bloque FedProx pilot.')
print('Recordatorio: este notebook NO lanza full FL y NO evalua downstream.')